In [4]:
import os
import time
from dynamixel_sdk import * # Uses Dynamixel SDK library

# --- Control Table Addresses for Dynamixel X-series ---
ADDR_TORQUE_ENABLE          = 64
ADDR_GOAL_POSITION          = 116
ADDR_PRESENT_POSITION       = 132

# --- Protocol version ---
PROTOCOL_VERSION            = 2.0

# --- Default setting ---
# OpenManipulator-X default IDs: 11, 12, 13, 14 (Joints) and 15 (Gripper)
DXL_IDs                     = [11, 12, 13, 14, 15] 
BAUDRATE                    = 1000000
DEVICENAME                  = 'COM9'  # MUST UPDATE THIS TO YOUR OPENCR COM PORT (check Device Manager)

TORQUE_ENABLE               = 1
TORQUE_DISABLE              = 0

# Initialize PortHandler and PacketHandler instance
portHandler = PortHandler(DEVICENAME)
packetHandler = PacketHandler(PROTOCOL_VERSION)

'''
# --- Commented out manual test code ---
def main():
    # Open port
    if portHandler.openPort():
        print("Succeeded to open the port")
    else:
        print("Failed to open the port")
        return

    # Set port baudrate
    if portHandler.setBaudRate(BAUDRATE):
        print("Succeeded to change the baudrate")
    else:
        print("Failed to change the baudrate")
        return

    # Enable Dynamixel Torque for all joints + gripper
    print("Enabling Torque...")
    for dxl_id in DXL_IDs:
        dxl_comm_result, dxl_error = packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_ENABLE)
        if dxl_comm_result != COMM_SUCCESS:
            print(f"ID {dxl_id}: %s" % packetHandler.getTxRxResult(dxl_comm_result))
        elif dxl_error != 0:
            print(f"ID {dxl_id}: %s" % packetHandler.getRxPacketError(dxl_error))
        else:
            print(f"Dynamixel ID {dxl_id} Torque Enabled")

    # Move Joint 1 (ID 11) to center position (2048)
    # Range is roughly 0 to 4095 for full rotation on X-series
    goal_position = 2048
    dxl_id_to_move = 11
    
    print(f"Moving ID {dxl_id_to_move} to position {goal_position}...")
    dxl_comm_result, dxl_error = packetHandler.write4ByteTxRx(portHandler, dxl_id_to_move, ADDR_GOAL_POSITION, goal_position)
    
    if dxl_comm_result != COMM_SUCCESS:
        print("%s" % packetHandler.getTxRxResult(dxl_comm_result))
    elif dxl_error != 0:
        print("%s" % packetHandler.getRxPacketError(dxl_error))

    time.sleep(2) # Wait for it to move

    # Disable Dynamixel Torque before exiting
    print("Disabling Torque...")
    for dxl_id in DXL_IDs:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_DISABLE)

    # Close port
    portHandler.closePort()
    print("Port closed.")

if __name__ == '__main__':
    main()
'''

'\n# --- Commented out manual test code ---\ndef main():\n    # Open port\n    if portHandler.openPort():\n        print("Succeeded to open the port")\n    else:\n        print("Failed to open the port")\n        return\n\n    # Set port baudrate\n    if portHandler.setBaudRate(BAUDRATE):\n        print("Succeeded to change the baudrate")\n    else:\n        print("Failed to change the baudrate")\n        return\n\n    # Enable Dynamixel Torque for all joints + gripper\n    print("Enabling Torque...")\n    for dxl_id in DXL_IDs:\n        dxl_comm_result, dxl_error = packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_ENABLE)\n        if dxl_comm_result != COMM_SUCCESS:\n            print(f"ID {dxl_id}: %s" % packetHandler.getTxRxResult(dxl_comm_result))\n        elif dxl_error != 0:\n            print(f"ID {dxl_id}: %s" % packetHandler.getRxPacketError(dxl_error))\n        else:\n            print(f"Dynamixel ID {dxl_id} Torque Enabled")\n\n    # Move Joint 1 (

In [5]:
# Helper functions to read and control joint positions

def connect_robot():
    """Opens the port and enables torque for all joints."""
    if not portHandler.is_open:
        portHandler.openPort()
        portHandler.setBaudRate(BAUDRATE)
        
    for dxl_id in DXL_IDs:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_ENABLE)
    print("Robot connected and Torque Enabled.")

def disconnect_robot():
    """Disables torque and closes the port."""
    for dxl_id in DXL_IDs:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_DISABLE)
    portHandler.closePort()
    print("Robot disconnected and Torque Disabled.")

def read_positions(verbose=True):
    """Reads and optionally prints the current position of all joints."""
    if verbose:
        print("--- Current Joint Positions ---")
    positions = {}
    for dxl_id in DXL_IDs:
        # Read 4 bytes for Present Position
        dxl_present_position, dxl_comm_result, dxl_error = packetHandler.read4ByteTxRx(portHandler, dxl_id, ADDR_PRESENT_POSITION)
        
        if dxl_comm_result != COMM_SUCCESS:
            if verbose:
                print(f"ID {dxl_id} Comm Error: {packetHandler.getTxRxResult(dxl_comm_result)}")
        elif dxl_error != 0:
            if verbose:
                print(f"ID {dxl_id} Packet Error: {packetHandler.getRxPacketError(dxl_error)}")
        else:
            if verbose:
                print(f"ID {dxl_id} \t Position: {dxl_present_position}")
            positions[dxl_id] = dxl_present_position
    return positions

def set_position(dxl_id, position):
    """Moves a specific joint (dxl_id) to a target position (0-4095)."""
    # Ensure position is within safe logical limits (0-4095 for X-series)
    position = max(0, min(4095, position)) 
    
    print(f"Moving ID {dxl_id} to {position}...")
    dxl_comm_result, dxl_error = packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, position)
    
    if dxl_comm_result != COMM_SUCCESS:
        print(f"Error: {packetHandler.getTxRxResult(dxl_comm_result)}")
    elif dxl_error != 0:
        print(f"Error: {packetHandler.getRxPacketError(dxl_error)}")

def set_positions_smooth(target_positions, steps=50, step_delay=0.02):
    """Moves joints smoothly to target positions using linear interpolation (gradient based)."""
    current_positions = read_positions(verbose=False)
    
    # Calculate per-step increments (the gradient for each joint)
    increments = {}
    for dxl_id in DXL_IDs:
        if dxl_id in current_positions and dxl_id in target_positions:
            start_pos = current_positions[dxl_id]
            end_pos = target_positions[dxl_id]
            increments[dxl_id] = (end_pos - start_pos) / steps
            
    # Execute steps sequentially to form a smooth transition
    for step in range(1, steps + 1):
        for dxl_id in DXL_IDs:
            if dxl_id in increments:
                # Calculate intermediate position
                intermediate_pos = int(current_positions[dxl_id] + (increments[dxl_id] * step))
                # Ensure it's within safe logical limits
                intermediate_pos = max(0, min(4095, intermediate_pos))
                
                # Write intermediate goal position silently
                packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, intermediate_pos)
        
        time.sleep(step_delay)


In [6]:
# Install ipywidgets if you don't have it already
# !pip install ipywidgets

import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output
import time

# UI Functions
def ui_connect(b):
    connect_robot()
    ui_read(None)

def ui_disconnect(b):
    disconnect_robot()

def ui_read(b):
    try:
        positions = read_positions(verbose=False)
        for i, dxl_id in enumerate(DXL_IDs):
            if dxl_id in positions:
                # Update visual labels
                read_labels[i].value = f" Actual: {positions[dxl_id]}"
    except Exception as e:
        print(f"Error reading: {e}")

def ui_apply(b):
    target_positions = {}
    for i, dxl_id in enumerate(DXL_IDs):
        target_positions[dxl_id] = sliders[i].value
        
    # Use the smooth, gradient-based transition function over 30 steps
    set_positions_smooth(target_positions, steps=30, step_delay=0.03)
    
    # Wait a short moment then update the actual readings
    time.sleep(0.2)
    ui_read(None)

def ui_default(b):
    # Reset all sliders to their default center position (2048)
    for slider in sliders:
        slider.value = 2048
    # Automatically apply the move
    ui_apply(None)

# Create GUI Elements
btn_connect = widgets.Button(description="1. Connect", button_style='info')
btn_connect.on_click(ui_connect)

btn_disconnect = widgets.Button(description="4. Disconnect", button_style='warning')
btn_disconnect.on_click(ui_disconnect)

btn_read = widgets.Button(description="2. Read Positions", button_style='primary')
btn_read.on_click(ui_read)

btn_apply = widgets.Button(description="3. Move Robot smoothly", button_style='success')
btn_apply.on_click(ui_apply)

btn_default = widgets.Button(description="Default Position", button_style='info')
btn_default.on_click(ui_default)

sliders = []
read_labels = []

# Friendly names for the OpenManipulator-X default joints
joint_names = ['Base (ID 11)', 'Shoulder (ID 12)', 'Elbow (ID 13)', 'Wrist (ID 14)', 'Gripper (ID 15)']

sliders_vbox_items = []

for i, dxl_id in enumerate(DXL_IDs):
    # Sliders for target position (0 to 4095 range for Dynamixel X-series)
    slider = widgets.IntSlider(value=2048, min=0, max=4095, step=1, 
                               description=f'ID {dxl_id}:', continuous_update=False)
    label = widgets.Label(value=f" Actual: Unknown", layout=widgets.Layout(width='150px'))
    name_label = widgets.Label(value=joint_names[i], layout=widgets.Layout(width='120px'))
    
    sliders.append(slider)
    read_labels.append(label)
    
    # Combine into one row per motor
    sliders_vbox_items.append(widgets.HBox([name_label, slider, label]))

# Combine everything into a nice layout
gui_layout = widgets.VBox([
    widgets.HTML("<h2>OpenManipulator-X Control Panel</h2>"),
    widgets.HBox([btn_connect, btn_disconnect]),
    widgets.HTML("<hr>"),
    widgets.VBox(sliders_vbox_items),
    widgets.HTML("<br>"),
    widgets.HBox([btn_read, btn_apply, btn_default])
])

# Display the UI
display(gui_layout)